 ### Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web or running code. Tools are pairing of :
  1. A schema,including the name of the tool, a description, and/or argument definitions(often a JSON schema)
  2. A function or coroutine to execute

In [8]:
import os 
from langchain.chat_models import init_chat_model
from langchain.tools import tool
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")
model=init_chat_model("google_genai:gemini-3.5-flash-lite")
@tool
def get_weather(location:str)->str:
    """Get a weather in a location."""
    return f" It's sunny in {location}"
model_with_tool=model.bind_tools([get_weather])



In [9]:
response=model_with_tool.invoke("What's the weather of germany?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool:{tool_call['name']}")
    print(f"Args:{tool_call['args']}")




content=[] additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Germany"}'}, '__gemini_function_call_thought_signatures__': {'call_2633405': 'EmAKXgFpFH0TjhucQSRWvyh0ggWVcj81LW5uwPzYW65K89QHQDGCSzhwMVBTm2MJ8aEGR0ZC8gRBUUupjNsDwocGfCgL6LpAdY7xCZdsBD7sn3pWMOqVMwBGmLXdCCptjTk='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a1781d-39df-7501-9675-e18fcacb4b15-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Germany'}, 'id': 'call_2633405', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 53, 'output_tokens': 16, 'total_tokens': 69, 'input_token_details': {'cache_read': 0}}
Tool:get_weather
Args:{'location': 'Germany'}


 ### Tool Execution Loops

In [12]:
# Model generates tool calls 
messages=[{"role":"user","content":"What's the weather in bhutan?"}]
ai_msgs=model_with_tool.invoke(messages)
messages.append(ai_msgs)
#Execute tools and collect results
for tool_call in ai_msgs.tool_calls:
    #execute the tool with the generated arguments
    tool_result=get_weather.invoke(tool_call)
    messages.append(tool_result)
#pass results back to model for final response
final_response=model_with_tool.invoke(messages)
print(final_response.text)



It's sunny in Bhutan!


 ### Messages 

Message are the fundamental unit of context  for models in langchain.They represent the input and output of models, carry both the content and metadata needed to represent the state of a conversation when interating with an LLM. Messages are objects that cantain:

1. Role- Identifies the message type(e.g. system, user)
2. Content- Represents the actual content of the message(like text ,images,audio,documents,etc)
3. MetaData- Optional field such as response information,message IDs and token usage

In [ ]:
import os 
from langchain.chat_models import init_chat_model
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")
model=init_chat_model("google_genai:gemini-3.5-flash")
model.invoke("best approach to learn LLM model as a beginner?")

AIMessage(content=[{'type': 'text', 'text': 'Learning Large Language Models (LLMs) as a beginner can feel overwhelming because the field spans everything from simple prompt writing to complex mathematics. \n\nThe best approach is a **"Top-Down" method**: start by using existing tools and building simple applications, then gradually look "under the hood" to understand how they work mathematically and architecturally.\n\nHere is a structured, step-by-step roadmap to learn LLMs effectively.\n\n---\n\n### Step 1: Understand the Core Concepts (No Coding Required)\nBefore writing code, get an intuitive understanding of what LLMs are and how they operate.\n\n*   **Key Concepts to Learn:**\n    *   **What is an LLM?** (Generative AI, Next-token prediction).\n    *   **Key Terminology:** Tokens, Context Window, Temperature, Hallucinations, System Prompts.\n    *   **Basic Mechanics:** Embeddings (how text turns into numbers) and Transformers (the core architecture).\n*   **Recommended Resources

 ### Use text prompts when:

1. You have a single, standalone request. 
2. You don't need conversation history. 
3. You want minimal code complexity.

In [ ]:
#Text Prompt :Text prompts are string.
model.invoke("What is langchain?")

AIMessage(content=[{'type': 'text', 'text': '**LangChain** is an open-source framework designed to help developers build applications powered by Large Language Models (LLMs), such as OpenAI’s GPT-4, Google’s Gemini, or open-source models like Llama.\n\nWhile an LLM on its own is just a text generator, **LangChain provides the "plumbing" to connect LLMs to other sources of data and computation**, allowing developers to create complex, context-aware AI applications.\n\n---\n\n### Why do we need LangChain?\n\nOn their own, LLMs have several limitations:\n1. **They don\'t know your private data:** An LLM doesn\'t know what is inside your company’s internal documents, Notion pages, or SQL databases.\n2. **They can\'t take actions:** An LLM can tell you *how* to book a flight, but it can\'t actually book it for you.\n3. **They lack memory:** Standard APIs for LLMs don\'t remember past conversations unless you manually resend the history every time.\n\nLangChain solves these problems by allow

Message Prompt: Alternatively,you can pass in a list of messages to the model by providing a list of message object.

 ### Message types

1. System Message - Tells the model how to behave and provide context for interactions

2. Human Message - Represents user input and interations with the model.

3. AI Message - Responses generated by the model,including text content ,tool calls and metadata.

4. Tool Message - Represents the output of tool calls.

In [ ]:
from langchain.messages import SystemMessage,HumanMessage,AIMessage
messages=[
    SystemMessage("You are a poetry expext"),
    HumanMessage("Write a poem on GenZ")
]
response=model.invoke(messages)
response.text

"They were born in the halo of a glass-tiled screen,\nWhere the air hums softly with a blue-light dream,\nSaddled with the centuries of all that came before,\nTold to build a future while the world keeps score.\n\nArchitects of irony, masters of the scroll,\nCarrying the weight of a warming planet's toll,\nThey doomscroll through the madness at three in the morn,\nUsing absurdity to mend where the fabric has torn.\n\nTheir language is fluid, a fast-shifting stream,\nCoded in emojis and thrift-store dreams,\nHyper-connected, yet searching for space,\nFinding profoundness in an un-filtered face.\n\nThey break down the boxes, the labels, the walls,\nIgnoring the echoes of traditional halls.\nSoft-hearted warriors of pronoun and grace,\nDefying the boundaries of time and of place.\n\nThey weep for the ocean, they march for the street,\nWhere the digital pulse and the pavement meet.\nTired of promises, weary of delay,\nThey burn down the old scripts to forge their own way.\n\nDo not mistake

In [ ]:
sysmsg=SystemMessage("Act like an senior AI Engineer expert")
messages=[
    sysmsg,
    HumanMessage("What are the primary requirement for a fresher for the AI Engineer Role?")
]
response=model.invoke(messages)
response.text

'As a Senior AI Engineer who has interviewed and mentored dozens of entry-level candidates, I can tell you that the market for AI roles has shifted dramatically. \n\nA few years ago, knowing how to train an XGBoost model or build a simple CNN in Keras was enough. Today, an **AI Engineer** is expected to be a **solid Software Engineer first**, and an AI specialist second. Companies don\'t just want someone who understands algorithms; they want someone who can build, integrate, and deploy AI-driven applications.\n\nHere is the realistic, industry-tested breakdown of what we look for in a fresher for an AI Engineer role, categorized from foundational to high-value differentiators.\n\n---\n\n### 1. Strong Software Engineering Fundamentals (The Non-Negotiable)\nThe biggest mistake freshers make is focusing 100% on AI math and 0% on writing clean code. 80% of AI engineering is traditional software engineering.\n\n* **Python Proficiency:** Advanced Python is mandatory (Object-Oriented Program

In [13]:
## Message Metadat
human_msg=HumanMessage(
    content="Human!",
    name="Alice",
    id="msg_123"
)
response=model.invoke([ human_msg])
response.text
response.usage_metadata


{'input_tokens': 3,
 'output_tokens': 9,
 'total_tokens': 12,
 'input_token_details': {'cache_read': 0}}